# Packages

In [1]:
import pickle
from datetime import datetime, timezone
from pathlib import Path
import tarfile

import boto3

# Loading Model and Scaler

In [2]:
with open("../model/model.pkl", "rb") as f:
    artifact = pickle.load(f)

model = artifact["model"]          # sklearn LogisticRegression
scaler = artifact["scaler"]        # sklearn StandardScaler
feature_cols = artifact["feature_cols"]  # ordem exata das features esperadas

# Registering Model and Scaler at Model Registry

## Defining Base Variables

In [3]:
REGION = "us-east-1"
ACCOUNT_ID = "272175292064" 
MODEL_PACKAGE_GROUP_NAME = "purchase-propensity-model-group"
MODEL_NAME = "purchase_propensity_v1"
MODEL_VERSION = "UAT"
MODEL_BUCKET = "personalization-models-272175292064"
MODEL_PREFIX = f"models/purchase_propensity/{MODEL_VERSION}"
INFERENCE_IMAGE_URI = (
    f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/personalization-model-train:{MODEL_VERSION}"
)
LOCAL_MODEL_DIR = Path("../model")

## Getting Clients

In [4]:
sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)

## Retrievering Model Locally

In [5]:
model_pkl = LOCAL_MODEL_DIR / "model.pkl"
if not model_pkl.exists():
    raise FileNotFoundError(f"Missing model file: {model_pkl}")
artifact_files = [p for p in LOCAL_MODEL_DIR.iterdir() if p.is_file() and p.name != "model.tar.gz"]
if not artifact_files:
    raise ValueError(f"No files found to package in {LOCAL_MODEL_DIR}")


## Creating Model TAR File

In [6]:
archive_path = LOCAL_MODEL_DIR / "model.tar.gz"
with tarfile.open(archive_path, "w:gz") as tar:
    for file_path in artifact_files:
        tar.add(file_path, arcname=file_path.name)

## Uploading Model TAR at S3

In [8]:
model_key = f"{MODEL_PREFIX}/model.tar.gz"
s3.upload_file(str(archive_path), MODEL_BUCKET, model_key)
model_data_url = f"s3://{MODEL_BUCKET}/{model_key}"

## Checking if Model Package Group Exists, if not Creating IT

In [9]:
try:
    sm.describe_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME)
except sm.exceptions.ClientError as e:
    if "ResourceNotFound" in str(e):
        sm.create_model_package_group(
            ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
            ModelPackageGroupDescription="Purchase propensity models generated by training pipeline."
        )
    else:
        raise

##  Registering Model at Group

In [10]:
description = (
    f"Purchase propensity model version {MODEL_VERSION} "
    f"registered manually at {datetime.now(timezone.utc).isoformat()}."
)
resp = sm.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    ModelPackageDescription=description,
    ModelApprovalStatus="Approved",
    InferenceSpecification={
        "Containers": [
            {
                "Image": INFERENCE_IMAGE_URI,
                "ModelDataUrl": model_data_url,
            }
        ],
        "SupportedContentTypes": ["application/json"],
        "SupportedResponseMIMETypes": ["application/json"],
    },
    CustomerMetadataProperties={
        "model_name": MODEL_NAME,
    },
)
print("ModelDataUrl:", model_data_url)
print("ModelPackageArn:", resp["ModelPackageArn"])

ModelDataUrl: s3://personalization-models-272175292064/models/purchase_propensity/UAT/model.tar.gz
ModelPackageArn: arn:aws:sagemaker:us-east-1:272175292064:model-package/purchase-propensity-model-group/1


## Checking if Model is Registered

In [11]:
sm.describe_model_package(ModelPackageName=resp["ModelPackageArn"])

{'ModelPackageGroupName': 'purchase-propensity-model-group',
 'ModelPackageVersion': 1,
 'ModelPackageRegistrationType': 'Registered',
 'ModelPackageArn': 'arn:aws:sagemaker:us-east-1:272175292064:model-package/purchase-propensity-model-group/1',
 'ModelPackageDescription': 'Purchase propensity model version UAT registered manually at 2026-07-29T15:36:56.831891+00:00.',
 'CreationTime': datetime.datetime(2026, 7, 29, 12, 36, 57, 541000, tzinfo=tzlocal()),
 'InferenceSpecification': {'Containers': [{'Image': '272175292064.dkr.ecr.us-east-1.amazonaws.com/personalization-model-train:UAT',
    'ImageDigest': 'sha256:4033edd79c12b714ebbba4a4b470a9e19b80b8f3fa3b88524b9e1ff6f0b35fe6',
    'ModelDataUrl': 's3://personalization-models-272175292064/models/purchase_propensity/UAT/model.tar.gz',
    'ModelDataETag': 'fa165937e10d24a02fea0554462969b6',
    'IsCheckpoint': False}],
  'SupportedContentTypes': ['application/json'],
  'SupportedResponseMIMETypes': ['application/json']},
 'ModelPackageS